# AI Toolkit ConvRot8 TPU feasibility probe

Enable the Kaggle **TPU** accelerator and Internet access. Attach `kaggle_tpu_probe_bundle.zip` as a private Kaggle input dataset, then run both cells. Kaggle may leave the ZIP intact or extract it automatically; both layouts work. The bundle contains only the probe and the fork's XLA ConvRot backend; it has no models, training data, tokens, or repository archive.

The notebook checks Kaggle's installed Torch/XLA versions before importing Torch or initializing PJRT. If either wheel is absent or incompatible, it installs the official candidate pair `torch==2.8.0` and `torch_xla[tpu]==2.8.0` into an isolated virtual environment under `/kaggle/working`; it does not alter the notebook kernel. That candidate is still subject to this actual TPU gate.

Download `diagnostic.json` and the neighboring log file from the printed `/kaggle/working/...` paths and share them for the next gate. A passing small projection proves only the runtime, collective, and ConvRot kernel path. The complete Qwen training fit remains pending until the exact model and cache artifacts are supplied and tested.

In [ ]:
import hashlib, importlib.metadata, json, os, pathlib, subprocess, sys, uuid, zipfile

zip_inputs = list(pathlib.Path('/kaggle/input').rglob('kaggle_tpu_probe_bundle.zip'))
manifest_inputs = list(pathlib.Path('/kaggle/input').rglob('source_manifest.json'))
if len(zip_inputs) == 1:
    with zipfile.ZipFile(zip_inputs[0]) as bundle:
        if set(bundle.namelist()) != {'probe.py', 'convrot_xla.py', 'source_manifest.json'}:
            raise RuntimeError('Unexpected ZIP members')
        files = {name: bundle.read(name) for name in bundle.namelist()}
elif len(zip_inputs) == 0 and len(manifest_inputs) == 1:
    folder = manifest_inputs[0].parent
    files = {name: (folder / name).read_bytes() for name in ('probe.py', 'convrot_xla.py', 'source_manifest.json')}
else:
    raise RuntimeError(f'Attach one probe bundle input; found {len(zip_inputs)} ZIPs and {len(manifest_inputs)} extracted manifests')
manifest = json.loads(files['source_manifest.json'])
if manifest.get('schema') != 1 or set(manifest.get('files', {})) != {'probe.py', 'convrot_xla.py'}:
    raise RuntimeError('Invalid or incomplete source manifest')
for name, digest in manifest['files'].items():
    if hashlib.sha256(files[name]).hexdigest() != digest:
        raise RuntimeError(f'Bundle checksum mismatch: {name}')
probe_dir = pathlib.Path('/kaggle/working') / ('ai_toolkit_tpu_probe_' + uuid.uuid4().hex)
probe_dir.mkdir()
for name, content in files.items(): (probe_dir / name).write_bytes(content)
print('Verified source commit:', manifest['source_commit'])
print('Verified source SHA-256:', manifest['files'])
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
versions = {name: version(name) for name in ('torch', 'torch-xla', 'libtpu')}
print('Installed runtime:', versions)
probe_python = sys.executable
matching = versions['torch'] and versions['torch-xla'] and versions['libtpu'] and versions['torch'].split('+')[0].split('.')[:2] == versions['torch-xla'].split('+')[0].split('.')[:2]
if not matching:
    if not ((3, 11) <= sys.version_info[:2] <= (3, 13)):
        raise RuntimeError('This notebook supports the pinned Torch/XLA 2.8 candidate on Python 3.11-3.13')
    venv_dir = pathlib.Path('/kaggle/working/ai_toolkit_tpu_probe_venv')
    subprocess.run([sys.executable, '-m', 'venv', str(venv_dir)], check=True)
    probe_python = str(venv_dir / 'bin/python')
    subprocess.run([probe_python, '-m', 'pip', 'install', '--no-cache-dir', 'torch==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
    subprocess.run([probe_python, '-m', 'pip', 'install', '--no-cache-dir', 'torch_xla[tpu]==2.8.0'], check=True)
    print('Installed official Torch/XLA 2.8 candidate in isolated venv:', probe_python)
print('Probe source directory:', probe_dir)

In [ ]:
import signal

report_dir = pathlib.Path('/kaggle/working') / ('ai_toolkit_tpu_report_' + uuid.uuid4().hex)
log_path = pathlib.Path(str(report_dir) + '.log')
env = os.environ.copy()
env['PJRT_DEVICE'] = 'TPU'
command = [probe_python, str(probe_dir / 'probe.py'), '--output', str(report_dir)]
timeout_minutes = 20
timed_out = False
with log_path.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=probe_dir, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    try:
        exit_code = process.wait(timeout=timeout_minutes * 60)
    except subprocess.TimeoutExpired:
        timed_out = True
        try: os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=10)
        except subprocess.TimeoutExpired: pass
        try: os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError: pass
        process.wait()
        exit_code = process.returncode
if timed_out:
    report_dir.mkdir(exist_ok=True)
    partial_ranks = [json.loads(path.read_text()) for path in sorted(report_dir.glob('rank-*.json'))]
    timeout_report = {'schema': 1, 'status': 'error', 'error': f'Probe exceeded {timeout_minutes} minutes; process group terminated', 'ranks': partial_ranks, 'completed_logical_devices': sum(r.get('status') == 'pass' for r in partial_ranks), 'source_manifest': manifest}
    (report_dir / 'diagnostic.json').write_text(json.dumps(timeout_report, indent=2))
diagnostic_path = report_dir / 'diagnostic.json'
print('Report directory:', report_dir)
print('Log file:', log_path)
print('Exit code:', exit_code)
if diagnostic_path.exists():
    diagnostic = json.loads(diagnostic_path.read_text())
    print('Status:', diagnostic['status'])
    print('Completed logical devices:', diagnostic['completed_logical_devices'])
    print('Expected logical devices:', diagnostic.get('expected_logical_devices'))
    print('Per-rank results:', [(r['rank'], r['status']) for r in diagnostic['ranks']])
    if diagnostic.get('error'): print('Error:', diagnostic['error'])
else:
    print('Launcher failed before diagnostic creation; inspect the printed log file')
print('Last log lines:')
print('\n'.join(log_path.read_text(errors='replace').splitlines()[-40:]))
if exit_code or timed_out:
    raise RuntimeError('TPU probe failed; download diagnostic.json and the printed log file for diagnosis')